# 10 — Control as Inference: Soft Q-learning vs Hard Q-learning

**The cleanest place to *watch* control-as-inference.** Casting RL as probabilistic inference
(Levine 2018) replaces the hard Bellman `max` with a temperature-controlled **soft-max**, and
makes the optimal policy a **Boltzmann distribution over Q-values**. This notebook shows that
on a tiny tabular gridworld where everything is exact and fast — no gym, no GPU.

We compare, on the *same* gridworld:

| | Hard (standard RL) | Soft (control-as-inference) |
|---|---|---|
| Value backup | $V(s)=\max_a Q(s,a)$ | $V_\alpha(s)=\alpha\,\log\sum_a e^{Q(s,a)/\alpha}$ |
| Policy | greedy: $\arg\max_a Q$ | Boltzmann: $\pi(a\mid s)\propto e^{Q(s,a)/\alpha}$ |
| Limit | — | $\alpha\to 0 \Rightarrow$ soft $\to$ hard |

The temperature $\alpha$ is the *inference temperature* — exactly SAC's entropy coefficient.
As $\alpha\to 0$ the soft value collapses to the hard value and the Boltzmann policy collapses
to greedy. As $\alpha$ grows, the policy spreads mass over all near-optimal actions.

> Companion to the **Control as Inference** note and to notebook **02 (DDPG→TD3→SAC)**, where the
> `−α·logπ` term in SAC's target is this exact soft backup, in deep/continuous form.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
ALPHA = 0.5   # default inference temperature used throughout
print('numpy', np.__version__)

## 1. A tiny gridworld

A `5x5` open grid. Start at the top-left `(0,0)`; a single **goal** at the bottom-right `(4,4)`
ends the episode. Four deterministic moves (up/down/left/right); bumping a wall keeps you in place.
Every move costs **−1** (a *living cost*), so the agent wants the **shortest path** — and Q-value
gaps between good and bad actions are on the order of 1, comparable to the temperature $\alpha$,
which is exactly what makes the Boltzmann structure visible.

Why open (no walls)? Because then there are **many equally-short paths** to the goal. A greedy
policy must pick *one* arbitrarily; the Boltzmann policy will split probability between the
equally-good actions (e.g. `right` vs `down` along the diagonal). That difference is the whole point.

In [ ]:
class GridWorld:
    """Open NxN grid, deterministic moves, single +1 terminal goal."""
    ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
    ARROWS = ['↑', '↓', '←', '→']

    def __init__(self, n=5, goal=(4, 4), gamma=0.99, step_reward=-1.0):
        self.n = n
        self.goal = goal
        self.gamma = gamma
        self.step_reward = step_reward
        self.nS = n * n
        self.nA = 4

    def s2rc(self, s):
        return divmod(s, self.n)

    def rc2s(self, r, c):
        return r * self.n + c

    def is_terminal(self, s):
        return self.s2rc(s) == self.goal

    def step_model(self, s, a):
        """Return (next_state, reward, done) for the deterministic transition."""
        if self.is_terminal(s):
            return s, 0.0, True
        r, c = self.s2rc(s)
        dr, dc = self.ACTIONS[a]
        nr, nc = r + dr, c + dc
        if not (0 <= nr < self.n and 0 <= nc < self.n):
            nr, nc = r, c  # bump wall -> stay
        ns = self.rc2s(nr, nc)
        done = (self.s2rc(ns) == self.goal)
        return ns, self.step_reward, done   # every move costs -1; goal ends the episode

env = GridWorld()
print('states', env.nS, 'actions', env.nA, 'goal', env.goal, 'gamma', env.gamma)

## 2. Hard vs soft value iteration

Because the model is known and tiny, we can solve both backups *exactly* by value iteration.

**Hard** (standard optimality):
$$Q(s,a)=r(s,a)+\gamma\,V(s'),\qquad V(s)=\max_a Q(s,a)$$

**Soft** (control-as-inference / max-entropy):
$$Q_\alpha(s,a)=r(s,a)+\gamma\,V_\alpha(s'),\qquad V_\alpha(s)=\alpha\,\log\sum_a e^{Q_\alpha(s,a)/\alpha}$$

The only change is `max` → `α·logsumexp(·/α)` (a smooth maximum). We use a numerically stable
log-sum-exp.

In [ ]:
def stable_logsumexp(x, axis=-1):
    m = np.max(x, axis=axis, keepdims=True)
    return (m + np.log(np.sum(np.exp(x - m), axis=axis, keepdims=True))).squeeze(axis)

def build_QV_step(env, V):
    """One sweep: compute Q(s,a) from current V using the known model."""
    Q = np.zeros((env.nS, env.nA))
    for s in range(env.nS):
        if env.is_terminal(s):
            continue
        for a in range(env.nA):
            ns, r, _ = env.step_model(s, a)
            Q[s, a] = r + env.gamma * V[ns]
    return Q

def hard_value_iteration(env, iters=200, tol=1e-10):
    V = np.zeros(env.nS)
    for _ in range(iters):
        Q = build_QV_step(env, V)
        Vnew = Q.max(axis=1)
        Vnew[[s for s in range(env.nS) if env.is_terminal(s)]] = 0.0
        if np.max(np.abs(Vnew - V)) < tol:
            V = Vnew; break
        V = Vnew
    return V, build_QV_step(env, V)

def soft_value_iteration(env, alpha, iters=400, tol=1e-10):
    V = np.zeros(env.nS)
    term = [s for s in range(env.nS) if env.is_terminal(s)]
    for _ in range(iters):
        Q = build_QV_step(env, V)
        Vnew = alpha * stable_logsumexp(Q / alpha, axis=1)
        Vnew[term] = 0.0
        if np.max(np.abs(Vnew - V)) < tol:
            V = Vnew; break
        V = Vnew
    return V, build_QV_step(env, V)

V_hard, Q_hard = hard_value_iteration(env)
V_soft, Q_soft = soft_value_iteration(env, alpha=ALPHA)
print(f'hard V[start] = {V_hard[0]:.4f}   (= -shortest-path cost, discounted)')
print(f'soft V[start] = {V_soft[0]:.4f}   (>= hard, optimism from the entropy bonus)')

## 3. Value functions side by side

Both increase toward the goal (bottom-right). The soft value is everywhere **≥** the hard value:
`α·logsumexp ≥ max`, an optimism that is the price of the entropy bonus. The gap is largest where
several actions are similarly good, and vanishes as $\alpha\to 0$.

In [ ]:
def plot_value(ax, env, V, title):
    grid = V.reshape(env.n, env.n)
    im = ax.imshow(grid, cmap='viridis')
    for s in range(env.nS):
        r, c = env.s2rc(s)
        ax.text(c, r, f'{V[s]:.2f}', ha='center', va='center',
                color='white', fontsize=8)
    gr, gc = env.goal
    ax.add_patch(plt.Rectangle((gc-0.5, gr-0.5), 1, 1, fill=False, edgecolor='red', lw=2))
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    return im

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_value(axes[0], env, V_hard, 'Hard  V(s) = max_a Q')
plot_value(axes[1], env, V_soft, f'Soft  V(s) = a logsumexp(Q/a),  a={ALPHA}')
plt.tight_layout(); plt.show()

## 4. The policy: greedy vs Boltzmann

- **Hard** policy: $\arg\max_a Q(s,a)$ — one arrow per state.
- **Soft** policy: $\pi(a\mid s)=\mathrm{softmax}_a(Q(s,a)/\alpha)$ — a distribution.

Along the diagonal, `right` and `down` are equally optimal, so the greedy policy picks one
arbitrarily while the **Boltzmann policy splits ~50/50** — you can see it in the per-cell
probabilities below.

In [ ]:
def boltzmann_policy(Q, alpha):
    logits = Q / alpha
    logits -= logits.max(axis=1, keepdims=True)
    p = np.exp(logits)
    return p / p.sum(axis=1, keepdims=True)

def plot_greedy(ax, env, Q, title):
    ax.imshow(np.zeros((env.n, env.n)), cmap='Greys', vmin=0, vmax=1)
    for s in range(env.nS):
        if env.is_terminal(s):
            continue
        r, c = env.s2rc(s)
        a = int(np.argmax(Q[s]))
        ax.text(c, r, env.ARROWS[a], ha='center', va='center', fontsize=16)
    gr, gc = env.goal
    ax.text(gc, gr, 'G', ha='center', va='center', fontsize=14, color='red')
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

def plot_boltzmann(ax, env, pi, title):
    # color each cell by policy entropy; annotate with P(right)/P(down)
    ent = -np.sum(pi * np.log(pi + 1e-12), axis=1).reshape(env.n, env.n)
    ax.imshow(ent, cmap='magma', vmin=0, vmax=np.log(env.nA))
    for s in range(env.nS):
        if env.is_terminal(s):
            continue
        r, c = env.s2rc(s)
        ax.text(c, r, f'→{pi[s,3]:.2f}\n↓{pi[s,1]:.2f}', ha='center', va='center',
                color='white', fontsize=6.5)
    gr, gc = env.goal
    ax.text(gc, gr, 'G', ha='center', va='center', fontsize=14, color='cyan')
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

pi_soft = boltzmann_policy(Q_soft, alpha=ALPHA)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_greedy(axes[0], env, Q_hard, 'Hard: greedy arg max')
plot_boltzmann(axes[1], env, pi_soft, f'Soft: Boltzmann a={ALPHA} (color=entropy)')
plt.tight_layout(); plt.show()
print('start-state soft policy [up,down,left,right]:', np.round(pi_soft[0], 3))
print('  -> mass concentrates on the two shortest-path actions (down, right)')

## 5. Temperature sweep: soft $\to$ hard as $\alpha\to 0$

The single knob $\alpha$ interpolates between the two regimes. We sweep it and track two things:

1. **Mean policy entropy** — how stochastic the Boltzmann policy is (0 = greedy, $\log 4$ = uniform).
2. **Value gap** $\frac1{|S|}\sum_s (V_\alpha(s)-V_{\text{hard}}(s))$ — the optimism of the soft backup.

The **value gap → 0** as $\alpha\to 0$: the inference view *contains* standard RL as its
zero-temperature limit. Entropy also drops sharply, but **floors slightly above 0** — states with
exactly-tied optimal actions (the symmetric shortest paths) stay ~50/50 at *any* temperature, since
$e^{Q_1/\alpha}/e^{Q_2/\alpha}=1$ whenever $Q_1=Q_2$. Genuine ties never break; only near-ties do.

In [ ]:
alphas = np.array([2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01])
mean_ent, val_gap = [], []
nonterm = [s for s in range(env.nS) if not env.is_terminal(s)]
for a in alphas:
    Va, Qa = soft_value_iteration(env, alpha=float(a))
    pi = boltzmann_policy(Qa, alpha=float(a))
    ent = -np.sum(pi[nonterm] * np.log(pi[nonterm] + 1e-12), axis=1)
    mean_ent.append(float(ent.mean()))
    val_gap.append(float((Va[nonterm] - V_hard[nonterm]).mean()))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(alphas, mean_ent, 'o-'); ax[0].set_xscale('log')
ax[0].axhline(np.log(env.nA), ls='--', c='gray', label='uniform (log 4)')
ax[0].set_xlabel('temperature α (log)'); ax[0].set_ylabel('mean policy entropy')
ax[0].set_title('Policy: greedy (α→0) → uniform (α↑)'); ax[0].legend()
ax[1].plot(alphas, val_gap, 'o-', color='crimson')
ax[1].set_xscale('log'); ax[1].set_yscale('log')
ax[1].set_xlabel('temperature α (log)'); ax[1].set_ylabel('mean (V_soft - V_hard), log')
ax[1].set_title('Soft value collapses to hard value as α→0')
plt.tight_layout(); plt.show()
print('alpha:', alphas.tolist())
print('entropy:', [round(x,3) for x in mean_ent])
print('valgap :', [round(x,4) for x in val_gap])

## 6. Learning version (no model): soft Q-learning vs Q-learning

Value iteration used the known model. In practice we *learn* $Q$ from sampled transitions. The
only change between the two algorithms is, again, the backup target:

- **Q-learning (hard):** $y = r + \gamma\,\max_{a'} Q(s',a')$, explore with $\epsilon$-greedy.
- **Soft Q-learning:** $y = r + \gamma\,\alpha\log\sum_{a'} e^{Q(s',a')/\alpha}$, explore with the
  Boltzmann policy itself (exploration is *built in* — no $\epsilon$ needed).

This is the tabular ancestor of SAC: SAC's critic target `r + γ(min Q − α logπ)` is the same soft
backup written for continuous actions with a sampled entropy estimate.

In [ ]:
def run_q_learning(env, soft, alpha=ALPHA, episodes=4000, lr=0.5, eps=0.1,
                   max_steps=100, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.nS, env.nA))
    steps_hist = []
    for ep in range(episodes):
        s = 0  # start top-left
        for t in range(max_steps):
            if soft:
                pi = boltzmann_policy(Q[s:s+1], alpha)[0]
                a = int(rng.choice(env.nA, p=pi))
            else:
                a = int(rng.integers(env.nA)) if rng.random() < eps else int(np.argmax(Q[s]))
            ns, r, done = env.step_model(s, a)
            if soft:
                target = r + (0.0 if done else env.gamma * alpha *
                              stable_logsumexp(Q[ns:ns+1] / alpha, axis=1)[0])
            else:
                target = r + (0.0 if done else env.gamma * Q[ns].max())
            Q[s, a] += lr * (target - Q[s, a])
            s = ns
            if done:
                break
        steps_hist.append(t + 1)
    return Q, np.array(steps_hist)

Qh, sh = run_q_learning(env, soft=False, seed=1)
Qs, ss = run_q_learning(env, soft=True, alpha=ALPHA, seed=1)

def smooth(x, k=100):
    return np.convolve(x, np.ones(k)/k, mode='valid')

plt.figure(figsize=(7, 4))
plt.plot(smooth(sh), label='Q-learning (hard, ε-greedy)')
plt.plot(smooth(ss), label='soft Q-learning (Boltzmann)')
opt = abs(env.goal[0]) + abs(env.goal[1])
plt.axhline(opt, ls='--', c='gray', label=f'shortest path = {opt}')
plt.xlabel('episode (smoothed)'); plt.ylabel('steps to goal')
plt.title('Both reach the goal; soft explores via its own policy'); plt.legend()
plt.tight_layout(); plt.show()

V_start_hard = float(Qh[0].max())
V_start_soft = float(ALPHA * stable_logsumexp(Qs[0:1] / ALPHA, axis=1)[0])
print(f'learned hard greedy V[start] : {V_start_hard:.4f}')
print(f'learned soft  V[start]       : {V_start_soft:.4f}')
print('learned soft start policy [u,d,l,r]:', np.round(boltzmann_policy(Qs[0:1], ALPHA)[0], 3))

## 7. Takeaways

- **One line of code separates them.** `max` (hard) vs `α·logsumexp(·/α)` (soft). Everything else
  — the env, the TD loop — is identical. Control-as-inference is that substitution, justified by a
  probabilistic-graphical-model derivation (optimality variables $O_t$, backward messages $\beta$).
- **$\alpha$ is the inference temperature.** $\alpha\to 0$ recovers ordinary RL exactly; larger
  $\alpha$ trades reward for entropy, spreading mass over near-optimal actions.
- **Exploration is built in.** The soft agent explores by *sampling its own Boltzmann policy* — no
  $\epsilon$-greedy bolt-on. This is the whole selling point of max-entropy RL.

### Where you have already seen this

| Notebook | Same idea, other guise |
|---|---|
| **02 DDPG→TD3→SAC** | SAC's `r + γ(min Q − α logπ)` target **is** this soft backup, continuous-action form. |
| **04 GRPO** / DPO | KL-to-reference gives $\pi\propto\pi_{\text{ref}}e^{r/\beta}$ — the same exp-tilted posterior. |
| **07 offline IQL** | advantage-weighted extraction $\pi\propto\pi_\beta e^{A/\beta}$ — Boltzmann, restricted to data. |
| **08 GFlowNet** | samples $\propto$ reward — the same 'reward as unnormalized probability' starting point. |

The methods that take the **hard `max`** — DQN, vanilla PPO without an entropy bonus, plain MCTS —
are the contrast class: they are the $\alpha\to 0$ corner of this same picture.

### When does soft actually help? (and why this toy hides it)

On *this* task the greedy policy on the **hard** value gets more return — and that is **by design**,
not a weakness. Hard maximizes return; soft maximizes return **+ α·entropy**, so it deliberately
leaves a little reward on the table to stay stochastic. Measured on its own objective, soft 'wins';
measured on pure return on a solved, known task, hard wins.

The move that reconciles this in practice: **train soft, act greedy.**

- SAC *learns* with the stochastic policy but *evaluates* with the deterministic mean action
  $\mu_\theta(s)$; many setups also anneal $\alpha\to 0$. Entropy is a **training-time regularizer**,
  not the deployed product — you get soft's learning benefits, then hard's performance at test.
- Soft's real wins are mostly **not** exploration: with neural-net function approximation the
  $\log\sum\exp$ backup gives **lower-variance, better-conditioned gradients** and avoids premature
  collapse — this is why SAC is far more stable / sample-efficient than DDPG (same family, softened
  objective). It is also **robust / risk-sensitive** (keeps mass on all near-optimal actions) and
  folds exploration into the objective (no $\epsilon$ to tune) — which matters in **sparse-reward,
  high-dimensional, stochastic** settings.
- This 5×5 grid has *none* of those stressors: deterministic, fully known, 25 tabular states, dense
  −1 signal everywhere. So you see the **mechanism** (`max` → `α logsumexp`) but not the **payoff**.

> **Rule of thumb.** Soft ≈ a better way to *train*; hard (greedy) ≈ a better way to *act at
> deployment*; modern methods do both. Pure hard RL is actually rare — PPO adds an entropy bonus and
> RLHF adds a KL-to-reference, both 'soft' regularizers, even when the final policy is taken greedily.